# BCG X Datathon — LivraisonCo
## 01 · Exploratory Data Analysis (EDA)

**Goal:** understand data coverage, sensor reliability, and traffic patterns for the three road segments before forecasting hourly flow and occupancy for **7–11 November 2026**.

**Workflow:** load → inspect → filter exact segments → assess missingness → analyze temporal patterns → check values → export.

## 1. Setup and data loading

The raw files are read without modification. Paths work when VS Code is launched either from the repository root or from `notebooks/`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "data").is_dir():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
OUTPUT_DIR = DATA_DIR / "processed"

FILES = {
    "Champs-Elysées": "champs_elysees.csv",
    "Convention": "convention.csv",
    "Saint-Pères": "saints_peres.csv",
}
TARGETS = ["Débit horaire", "Taux d'occupation"]
TIME_COL = "Date et heure de comptage"
UPSTREAM = "Libelle noeud amont"
DOWNSTREAM = "Libelle noeud aval"

data = {
    road: pd.read_csv(DATA_DIR / filename, sep=";")
    for road, filename in FILES.items()
}
display(pd.DataFrame({
    "road": list(data),
    "rows": [len(df) for df in data.values()],
    "columns": [df.shape[1] for df in data.values()],
}))

## 2. Initial inspection

Review the original columns, sample records, and all-column missingness **before filtering**.

In [ ]:
for road, df in data.items():
    print(f"\n=== {road} ===")
    print("Columns:", df.columns.tolist())
    display(df.head(3))
    display(
        df.isna().mean().mul(100).round(2)
          .rename("missing_pct").to_frame().T
    )

## 3. Select the three official road segments

Each road is defined by an **exact upstream/downstream node pair**. Convention and Saints-Pères raw files contain multiple segments; we retain only the requested direction.

In [ ]:
TARGET_SEGMENTS = {
    "Champs-Elysées": ("Av_Champs_Elysees-Washington", "Av_Champs_Elysees-Berri"),
    "Convention": ("Lecourbe-Convention", "Convention-Blomet"),
    "Saint-Pères": ("Sts_Peres-Voltaire", "Sts_Peres-Universite"),
}

for road, df in data.items():
    print(f"\n=== Available segments: {road} ===")
    display(df[[UPSTREAM, DOWNSTREAM]].drop_duplicates().reset_index(drop=True))

data_filtered = {}
for road, df in data.items():
    upstream, downstream = TARGET_SEGMENTS[road]
    mask = df[UPSTREAM].eq(upstream) & df[DOWNSTREAM].eq(downstream)
    subset = df.loc[mask].copy()
    if subset.empty:
        raise ValueError(f"No records match the official segment for {road}")
    data_filtered[road] = subset

display(pd.DataFrame({
    "road": list(data_filtered),
    "raw_rows": [len(data[r]) for r in data_filtered],
    "selected_rows": [len(df) for df in data_filtered.values()],
}))

## 4. Data availability and temporal coverage

Convert timestamps once to **Europe/Paris local time**, retaining the original timestamp column. UTC-aware parsing handles offset changes. We report missingness separately for each target and for both together.

In [ ]:
for road, df in data_filtered.items():
    df["datetime"] = pd.to_datetime(
        df[TIME_COL], errors="coerce", utc=True
    ).dt.tz_convert("Europe/Paris")
    df.sort_values("datetime", inplace=True)

summary = []
for road, df in data_filtered.items():
    both = df[TARGETS].notna().all(axis=1)
    summary.append({
        "road": road,
        "rows": len(df),
        "first_timestamp": df["datetime"].min(),
        "last_timestamp": df["datetime"].max(),
        "unique_timestamps": df["datetime"].nunique(),
        "duplicate_timestamps": df["datetime"].duplicated().sum(),
        "flow_available": df[TARGETS[0]].notna().sum(),
        "occupancy_available": df[TARGETS[1]].notna().sum(),
        "both_available": both.sum(),
        "both_available_pct": round(100 * both.mean(), 1),
    })
display(pd.DataFrame(summary))

In [ ]:
last_valid = []
for road, df in data_filtered.items():
    for target in TARGETS:
        valid = df.loc[df[target].notna(), "datetime"]
        last_valid.append({
            "road": road, "target": target,
            "first_valid": valid.min(),
            "last_valid": valid.max(),
        })
display(pd.DataFrame(last_valid))

### 4.1 Monthly measurement availability

Percentage of existing hourly records with a non-missing **traffic flow** value. A separate timestamp-continuity check is still needed before modeling.

In [ ]:
for road, df in data_filtered.items():
    temp = df.copy()
    temp["month"] = temp["datetime"].dt.strftime("%Y-%m")
    monthly = temp.groupby("month", sort=True)[TARGETS[0]].apply(
        lambda s: 100 * s.notna().mean()
    )
    ax = monthly.plot.bar(figsize=(11, 3.2), title=f"Flow availability — {road}")
    ax.set(xlabel="Month", ylabel="Available measurements (%)", ylim=(0, 105))
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 5. Traffic patterns

All time-based features use **Paris local time**. Mean profiles exclude missing target measurements automatically. Champs-Élysées observations are largely restricted to October–November 2025, so comparisons with the other roads are not like-for-like.

In [ ]:
for df in data_filtered.values():
    df["hour"] = df["datetime"].dt.hour
    df["day_of_week"] = df["datetime"].dt.dayofweek
    df["is_weekend"] = df["day_of_week"] >= 5

def plot_targets(profiles, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    for ax, target in zip(axes, TARGETS):
        profiles[target].plot(ax=ax, marker="o")
        ax.set(title=f"{title} — {target}", xlabel="Hour")
        ax.grid(alpha=0.25)
    axes[0].set_ylabel("Vehicles / hour")
    axes[1].set_ylabel("Occupancy (%)")
    plt.tight_layout()
    plt.show()

### 5.1 Average profile by hour

In [ ]:
for road, df in data_filtered.items():
    hourly = df.groupby("hour")[TARGETS].mean().reindex(range(24))
    plot_targets(hourly, road)

### 5.2 Weekday versus weekend

Compare traffic profiles by hour; these averages do not control for seasonal coverage differences.

In [ ]:
for road, df in data_filtered.items():
    hourly = df.groupby(["is_weekend", "hour"])[TARGETS].mean()
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    for weekend, label in [(False, "Weekday"), (True, "Weekend")]:
        if weekend not in hourly.index.get_level_values(0):
            continue
        profile = hourly.loc[weekend].reindex(range(24))
        for ax, target in zip(axes, TARGETS):
            ax.plot(profile.index, profile[target], label=label)
    for ax, target in zip(axes, TARGETS):
        ax.set(title=f"{road} — {target}", xlabel="Hour")
        ax.legend()
        ax.grid(alpha=0.25)
    axes[0].set_ylabel("Vehicles / hour")
    axes[1].set_ylabel("Occupancy (%)")
    plt.tight_layout()
    plt.show()

### 5.3 Average profile by day of week

In [ ]:
DAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
for road, df in data_filtered.items():
    daily = df.groupby("day_of_week")[TARGETS].mean().reindex(range(7))
    daily.index = DAY_NAMES
    axes = daily.plot.bar(subplots=True, layout=(1, 2), figsize=(12, 3.5),
                          legend=False, rot=0, title=[f"{road} — Flow", f"{road} — Occupancy"])
    axes[0, 0].set_ylabel("Vehicles / hour")
    axes[0, 1].set_ylabel("Occupancy (%)")
    plt.tight_layout()
    plt.show()

## 6. Numerical quality checks

Flag impossible values, but **do not automatically delete** high measurements or zero occupancy: these may be genuine.

In [ ]:
quality = []
for road, df in data_filtered.items():
    flow = df[TARGETS[0]]
    occupancy = df[TARGETS[1]]
    quality.append({
        "road": road,
        "flow_mean": flow.mean(),
        "flow_min": flow.min(),
        "flow_max": flow.max(),
        "occupancy_mean_pct": occupancy.mean(),
        "occupancy_min_pct": occupancy.min(),
        "occupancy_max_pct": occupancy.max(),
        "negative_flow": (flow < 0).sum(),
        "zero_flow": flow.eq(0).sum(),
        "invalid_occupancy": ((occupancy < 0) | (occupancy > 100)).sum(),
    })
display(pd.DataFrame(quality).round(2))

## 7. Export selected segments

Save a reproducible, **unimputed** copy for the next notebook. Original raw files remain untouched.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for road, df in data_filtered.items():
    output_path = OUTPUT_DIR / FILES[road]
    # Export original fields; derived EDA columns will be rebuilt in preprocessing.
    export_cols = [c for c in data[road].columns if c in df.columns]
    df[export_cols].to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

## 8. EDA findings and next steps

**Findings to retain for the BCG X presentation**
- Champs-Élysées has a major sensor-availability issue: valid measurements stop in November 2025.
- Convention and Saints-Pères have much more recent valid measurements.
- Hourly patterns and weekday/weekend differences motivate calendar features.

**Before training**
1. Check missing **timestamps** (not just missing target values), including daylight-saving transitions.
2. Decide how to handle the Champs-Élysées sensor gap and whether fresher data can be obtained.
3. Use chronological train/validation splits and compare against a simple hour × weekday baseline.
4. Never use future target measurements as forecasting features.

> No RMSE or future forecast is claimed in this EDA notebook.